# UK Train Rides: Data Profiling

This notebook explores the raw railway dataset before any cleaning. The goal is to understand the structure of the data and find the quality issues we need to fix.

Nothing in the data is changed here. All fixes are applied in the next notebook, `02_cleaning_and_transformation.ipynb`.

## 1. Load the data

We import pandas and read the raw CSV file from the `data` folder.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/railway.csv")
df.shape

(31653, 18)

The dataset has **31,653 rows and 18 columns**. According to the data dictionary, each row is one ticket purchase, identified by `Transaction ID`.

Let's look at the first five rows to see what the values look like.

In [2]:
df.head()

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,Arrival Destination,Date of Journey,Departure Time,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Refund Request
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,Online,Contactless,Adult,Standard,Advance,43,London Paddington,Liverpool Lime Street,2024-01-01,11:00:00,13:30:00,13:30:00,On Time,NaN,No
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,Station,Credit Card,Adult,Standard,Advance,23,London Kings Cross,York,2024-01-01,09:45:00,11:35:00,11:40:00,Delayed,Signal Failure,No
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,Online,Credit Card,NaN,Standard,Advance,3,Liverpool Lime Street,Manchester Piccadilly,2024-01-02,18:15:00,18:45:00,18:45:00,On Time,NaN,No
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,Station,Credit Card,NaN,Standard,Advance,13,London Paddington,Reading,2024-01-01,21:30:00,22:30:00,22:30:00,On Time,NaN,No
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,Online,Contactless,NaN,Standard,Advance,76,Liverpool Lime Street,London Euston,2024-01-01,16:45:00,19:00:00,19:00:00,On Time,NaN,No


Dates and times are stored as text, and `Price` is a whole number in pounds. Some cells in `Railcard`, `Actual Arrival Time` and `Reason for Delay` are empty (`NaN`).

## 2. Column types

We check the data type pandas assigned to each column.

In [3]:
df.dtypes

Transaction ID           str
Date of Purchase         str
Time of Purchase         str
Purchase Type            str
Payment Method           str
Railcard                 str
Ticket Class             str
Ticket Type              str
Price                  int64
Departure Station        str
Arrival Destination      str
Date of Journey          str
Departure Time           str
Arrival Time             str
Actual Arrival Time      str
Journey Status           str
Reason for Delay         str
Refund Request           str
dtype: object

All columns are read as text (`str`) except `Price`, which is an integer. The two date columns and the four time columns will be converted to proper date and time types during cleaning.

## 3. Missing values

We count the empty cells in each column and keep only the columns that have at least one.

In [4]:
missing = df.isna().sum()
missing[missing > 0]

Railcard               20918
Actual Arrival Time     1880
Reason for Delay       27481
dtype: int64

Only three columns have missing values:

- `Railcard`: 20,918 rows. The data dictionary says passengers without a railcard are "None", so an empty value means the passenger has no railcard. These rows are valid and must be kept.
- `Actual Arrival Time`: 1,880 rows.
- `Reason for Delay`: 27,481 rows.

The last two might be related to the journey status. We check that next.

We compare `Journey Status` with whether `Actual Arrival Time` is empty.

In [5]:
pd.crosstab(df["Journey Status"], df["Actual Arrival Time"].isna(),
            colnames=["Actual Arrival Time is missing"])

Actual Arrival Time is missing,False,True
Journey Status,,
Cancelled,0,1880
Delayed,2292,0
On Time,27481,0


All 1,880 missing actual arrival times belong to **cancelled** journeys, and every cancelled journey has no actual arrival time. This makes sense: a cancelled train never arrives. These values should stay empty and must not be filled with made-up times.

We do the same check for `Reason for Delay`.

In [6]:
pd.crosstab(df["Journey Status"], df["Reason for Delay"].isna(),
            colnames=["Reason for Delay is missing"])

Reason for Delay is missing,False,True
Journey Status,,
Cancelled,1880,0
Delayed,2292,0
On Time,0,27481


All 27,481 missing reasons belong to **on-time** journeys, and every delayed or cancelled journey has a reason. So an empty reason simply means "no delay".

## 4. Duplicates

`Transaction ID` should be unique. We also check for rows that are fully repeated.

In [7]:
print("Duplicate Transaction IDs:", df["Transaction ID"].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

Duplicate Transaction IDs: 0
Fully duplicated rows: 0


There are no duplicate transaction IDs and no repeated rows, so `Transaction ID` can be used as the primary key of the ticket sales table.

## 5. Categorical columns

We list the distinct values and their counts for each text column with a small number of categories. This helps us spot spelling differences.

In [8]:
categorical_cols = ["Purchase Type", "Payment Method", "Railcard", "Ticket Class",
                    "Ticket Type", "Journey Status", "Reason for Delay", "Refund Request"]

for col in categorical_cols:
    print(df[col].value_counts(dropna=False), "\n")

Purchase Type
Online     18521
Station    13132
Name: count, dtype: int64 

Payment Method
Credit Card    19136
Contactless    10834
Debit Card      1683
Name: count, dtype: int64 

Railcard
NaN         20918
Adult        4846
Disabled     3089
Senior       2800
Name: count, dtype: int64 

Ticket Class
Standard       28595
First Class     3058
Name: count, dtype: int64 

Ticket Type
Advance     17561
Off-Peak     8752
Anytime      5340
Name: count, dtype: int64 

Journey Status
On Time      27481
Delayed       2292
Cancelled     1880
Name: count, dtype: int64 

Reason for Delay
NaN                   27481
Weather                 995
Technical Issue         707
Signal Failure          523
Signal failure          447
Staffing                410
Staff Shortage          399
Weather Conditions      377
Traffic                 314
Name: count, dtype: int64 

Refund Request
No     30535
Yes     1118
Name: count, dtype: int64 



Most columns are clean. The problem is in `Reason for Delay`, where the same reason is written in different ways:

- `Signal Failure` (523) and `Signal failure` (447)
- `Weather` (995) and `Weather Conditions` (377)
- `Staffing` (410) and `Staff Shortage` (399)

Each pair describes the same reason (this is also noted in the project schema), so they will be merged during cleaning. After merging there are five real reasons: Weather, Signal Failure, Staffing, Technical Issue and Traffic.

Other observations:

- Online purchases (18,521) are more common than station purchases (13,132).
- Most tickets are Standard class (28,595) and the most common ticket type is Advance (17,561).
- 1,118 tickets have a refund request.

## 6. Dates and times

We convert the date columns to datetime in separate variables, only to check them. The original `df` is not modified.

In [9]:
purchase_date = pd.to_datetime(df["Date of Purchase"], format="%Y-%m-%d")
journey_date = pd.to_datetime(df["Date of Journey"], format="%Y-%m-%d")

print("Purchase dates:", purchase_date.min().date(), "to", purchase_date.max().date())
print("Journey dates: ", journey_date.min().date(), "to", journey_date.max().date())

Purchase dates: 2023-12-08 to 2024-04-30
Journey dates:  2024-01-01 to 2024-04-30


Journeys run from **1 January 2024 to 30 April 2024**. Purchases start earlier, on **8 December 2023**, because some tickets were bought weeks ahead. This matters for the date dimension: it must cover both ranges.

Next we calculate how many days before the journey each ticket was bought.

In [10]:
days_ahead = (journey_date - purchase_date).dt.days

print(days_ahead.describe().round(1))
print("\nTickets bought after the journey date:", (days_ahead < 0).sum())

count    31653.0
mean         1.7
std          3.9
min          0.0
25%          0.0
50%          1.0
75%          1.0
max         28.0
dtype: float64

Tickets bought after the journey date: 0


No ticket was bought after its journey date, so there are no invalid booking dates. Tickets were bought between 0 and 28 days ahead. The average is 1.7 days, and half of the tickets were bought 1 day or less before the journey, so most passengers book very close to travel. This value will become the `Days_Booked_Ahead` column.

The data dictionary says Advance tickets must be bought at least one day before departure. We check if the data follows this rule.

In [11]:
advance_same_day = (df["Ticket Type"] == "Advance") & (days_ahead < 1)
print("Advance tickets bought on the journey day:", advance_same_day.sum())

Advance tickets bought on the journey day: 0


The data follows the rule. There are no Advance tickets bought on the same day.

The dictionary also says arrival times can be on the day after departure. We look for rows where the scheduled arrival time is earlier than the departure time, which means the train arrives after midnight.

In [12]:
departure = pd.to_timedelta(df["Departure Time"])
scheduled_arrival = pd.to_timedelta(df["Arrival Time"])
actual_arrival = pd.to_timedelta(df["Actual Arrival Time"])

overnight = scheduled_arrival < departure
print("Tickets with arrival after midnight:", overnight.sum())
df.loc[overnight, ["Departure Station", "Arrival Destination",
                   "Departure Time", "Arrival Time"]].head()

Tickets with arrival after midnight: 961


,Departure Station,Arrival Destination,Departure Time,Arrival Time
34,London Euston,Birmingham New Street,23:30:00,00:50:00
92,Liverpool Lime Street,Manchester Piccadilly,23:30:00,00:00:00
94,London Kings Cross,York,23:45:00,01:35:00
95,York,Durham,23:45:00,00:35:00
97,York,Durham,23:45:00,00:35:00


961 tickets are for trains that arrive after midnight, for example the London Euston to Birmingham New Street train that leaves at 23:30 and arrives at 00:50. If we subtract clock times without considering the date change, the journey duration would come out negative. We need to keep this in mind when calculating delays.

Now we calculate the delay in minutes for delayed journeys by simply subtracting the scheduled arrival from the actual arrival.

In [13]:
delay_minutes = (actual_arrival - scheduled_arrival).dt.total_seconds() / 60
delayed = df["Journey Status"] == "Delayed"

print(delay_minutes[delayed].describe().round(1))
print("\nNegative delays:", (delay_minutes[delayed] < 0).sum())
print("Delayed tickets with 0 minutes delay:", (delay_minutes[delayed] == 0).sum())

count    2292.0
mean       42.2
std        34.5
min         0.0
25%        19.0
50%        37.0
75%        53.0
max       180.0
dtype: float64

Negative delays: 0
Delayed tickets with 0 minutes delay: 18


Delays range from 0 to 180 minutes, with an average of about 42 minutes. No delay is negative, which means no delayed train in this data crosses midnight between its scheduled and actual arrival. We will still handle the midnight case in the cleaning code so the calculation is always correct.

There are **18 tickets marked as Delayed with a 0-minute delay**. This is a small inconsistency in the source data. We will keep these rows as they are and document them instead of changing the status.

## 7. Price

We look at the price distribution and check for invalid values.

In [14]:
df["Price"].describe().round(2)

count    31653.00
mean        23.44
std         30.00
min          1.00
25%          5.00
50%         11.00
75%         35.00
max        267.00
Name: Price, dtype: float64

Prices go from £1 to £267, with an average of £23.44. There are no zero or negative prices.

We compare the average price across ticket classes and ticket types.

In [15]:
df.groupby(["Ticket Class", "Ticket Type"])["Price"].agg(["count", "mean", "min", "max"]).round(2)

count   mean  min  max
Ticket Class Ticket Type                        
First Class  Advance       1764  37.92    4  135
             Anytime        490  77.23    8  267
             Off-Peak       804  55.56    6  203
Standard     Advance      15797  15.34    1   86
             Anytime       4850  35.35    2  171
             Off-Peak      7948  22.48    2  128

The prices behave as expected. First Class is more expensive than Standard, and within each class Anytime is the most expensive, then Off-Peak, then Advance. This matches the discounts in the data dictionary (Advance 50% off, Off-Peak 25% off).

Because `Price` is the final cost, these discounts are already included in the price. We must not apply them again.

We record the total ticket value so we can check later that cleaning did not lose or change any money.

In [16]:
print("Total ticket value: £", df["Price"].sum())

Total ticket value: £ 741921


The total recorded ticket value is **£741,921**. The cleaned ticket table must give exactly the same total.

## 8. Tickets vs journeys

One row is one ticket, but many tickets can be for the same train journey. First we count the stations and routes.

In [17]:
stations = sorted(set(df["Departure Station"]) | set(df["Arrival Destination"]))
routes = df[["Departure Station", "Arrival Destination"]].drop_duplicates()

print("Stations:", len(stations))
print("Routes:", len(routes))
print(stations)

Stations: 32
Routes: 65
['Birmingham New Street', 'Bristol Temple Meads', 'Cardiff Central', 'Coventry', 'Crewe', 'Didcot', 'Doncaster', 'Durham', 'Edinburgh', 'Edinburgh Waverley', 'Leeds', 'Leicester', 'Liverpool Lime Street', 'London Euston', 'London Kings Cross', 'London Paddington', 'London St Pancras', 'London Waterloo', 'Manchester Piccadilly', 'Nottingham', 'Nuneaton', 'Oxford', 'Peterborough', 'Reading', 'Sheffield', 'Stafford', 'Swindon', 'Tamworth', 'Wakefield', 'Warrington', 'Wolverhampton', 'York']


There are **32 stations and 65 routes**. Note that `Edinburgh` and `Edinburgh Waverley` appear as two separate names. They are probably the same place, but the project schema counts 32 stations, so we keep them separate.

The project schema (`docs/UK_Train_Rides_Schema.docx`) defines a journey as a unique combination of route, journey date and departure time. We count how many journeys exist.

In [18]:
journey_key = ["Departure Station", "Arrival Destination", "Date of Journey", "Departure Time"]

print("Distinct journeys:", df[journey_key].drop_duplicates().shape[0])

Distinct journeys: 19871


The 31,653 tickets belong to **19,871 distinct journeys**. This is why the schema has two fact tables: one for tickets and one for journeys. Counting delays or cancellations on the ticket table would count the same journey many times.

Before splitting the data, we must confirm that all tickets of the same journey have the same journey details. We count the distinct values of each journey column inside every journey.

In [19]:
journey_cols = ["Arrival Time", "Actual Arrival Time", "Journey Status", "Reason for Delay"]

values_per_journey = df.groupby(journey_key)[journey_cols].nunique(dropna=False)
(values_per_journey > 1).sum()

Arrival Time           0
Actual Arrival Time    0
Journey Status         0
Reason for Delay       1
dtype: int64

Scheduled arrival, actual arrival and status are consistent inside every journey. Only one journey has two different values in `Reason for Delay`. We look at it.

In [20]:
conflict = values_per_journey[values_per_journey["Reason for Delay"] > 1].index
conflict_rows = df.set_index(journey_key).loc[conflict]
conflict_rows[["Journey Status", "Reason for Delay"]].value_counts()

Journey Status  Reason for Delay
Cancelled       Signal failure      7
                Signal Failure      1
Name: count, dtype: int64

It is a cancelled journey with 8 tickets. Seven tickets say `Signal failure` and one says `Signal Failure`, so the difference is only in spelling. After the reasons are standardized in cleaning, this journey becomes consistent too.

Finally we check `Refund Request`. It is a ticket-level field, because each passenger decides to ask for a refund or not.

In [21]:
refund_per_journey = df.groupby(journey_key)["Refund Request"].nunique()
print("Journeys with both Yes and No refund requests:", (refund_per_journey > 1).sum())

Journeys with both Yes and No refund requests: 383


383 journeys have some tickets with a refund request and others without. This confirms that `Refund Request` belongs in the ticket table, not the journey table, which is how the schema places it.

## 9. Summary of findings

- The data has 31,653 tickets and 18 columns. There are no duplicate transaction IDs.
- The tickets belong to 19,871 journeys, on 65 routes between 32 stations. Journey details are consistent across tickets of the same journey.
- Journey dates run from 1 January to 30 April 2024. Purchase dates start on 8 December 2023.
- Missing values are meaningful, not errors:
  - Empty `Railcard` (20,918) means no railcard and will become "None".
  - Empty `Actual Arrival Time` (1,880) happens only for cancelled journeys and will stay empty.
  - Empty `Reason for Delay` (27,481) happens only for on-time journeys and means no delay.
- `Reason for Delay` has three pairs of labels that mean the same thing. They will be merged into five reasons.
- 961 tickets arrive after midnight. Delay calculation must handle the day change.
- 18 tickets are marked Delayed with a 0-minute delay. They will be kept and documented.
- Prices are valid (£1 to £267) and already include discounts. The total ticket value is £741,921 and must stay the same after cleaning.
- `Refund Request` varies between tickets of the same journey, so it stays in the ticket table.

No rows need to be removed. The next notebook will clean the data and build the 11 tables of the schema.